# S01 Solar Power Day Ahead Forecasting
## Progress implementation with chronological validation

**Plant:** S01 (anonymized), 6,000 kW  
**Data source:** Provincial Electricity Authority  
**Study period:** 1 January 2023 to 31 December 2024  
**Resolution:** 5-minute observations aggregated to 96 daily 15-minute targets

This notebook covers loading, completeness auditing, preprocessing, forecasting
baselines, Ridge Regression (L2-regularized Linear Regression), XGBoost,
feature ablation, monthly evaluation, and prediction intervals.

**Rerun required:** outputs from the earlier implementation have been cleared.
Restart the kernel and Run All with the original Excel workbooks to produce
results for the corrected methodology. Earlier numerical results must not be
reported as results of this revision.

Forecasts are issued after the final 23:55 reading of D-1 is available. Inputs
use D-1 or earlier observations and known calendar fields. Monthly XGBoost
parameters are selected once using the initial January-June 2023 period and
then frozen for the July 2023 onward evaluation.


## 0 Setup

Keep this notebook in `notebooks/` and the two helper modules in `src/`.
Place the original monthly workbooks in `data/S01/2023/` and `data/S01/2024/`.
Run from the project root or the notebooks folder. Paths are resolved from the
current working directory; source measurements are not included in this bundle.
Generated tables, audit logs, and figures are written to `outputs/`.


In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn xgboost openpyxl


In [ ]:
# Use the Python environment selected by this Jupyter kernel.
# Restart the kernel after installing or replacing project modules.


In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


warnings.filterwarnings("ignore", category=FutureWarning)

PLANT_CAPACITY_KW = 6000
STUDY_START = "2023-01-01"
STUDY_END = "2024-12-31"
XGB_SEARCH_ITER = 30
# Optional confirmed exclusions: date -> documented reason. Empty by default.
MANUAL_EXCLUSIONS = {}
STEPS_PER_DAY = 96
RANDOM_STATE = 42

# === Paths ===
# ระบุรากโปรเจกต์จากโฟลเดอร์ที่เปิด notebook เพื่อให้ใช้ได้กับทุกเครื่อง
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists() and (REPO_ROOT.parent / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reload once at setup so all later cells share the same module/class versions.
import importlib
import src.data_quality as quality_module
quality_module = importlib.reload(quality_module)
import src.modeling as modeling_module
modeling_module = importlib.reload(modeling_module)

# Raw PEA Excel files (local only, gitignored)
DATA_ROOT = REPO_ROOT / "data" / "S01"
YEAR_FOLDERS = [DATA_ROOT / "2023", DATA_ROOT / "2024"]

# Cache file (speeds up reloading)
CACHE_FILE = REPO_ROOT / "data" / "S01_2023_2024_5min_cache_v3.csv"

# All figures and tables go here (tracked by git)
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("CACHE_FILE:", CACHE_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)


# 1 Data loading

Each monthly workbook contains daily worksheets with 5-minute measurements.
The loader uses the workbook name for month/year and the worksheet name for
the day. Invalid calendar dates are skipped. Cell A1 is not used for dating.


In [ ]:
# List all monthly Excel files from 2023 and 2024
excel_files = []

for folder in YEAR_FOLDERS:
    files = sorted(folder.glob("S01-*.xlsx"))
    excel_files.extend(files)
    print(f"{folder.name}: {len(files)} file(s)")

print("\nTotal Excel files:", len(excel_files))

if len(excel_files) == 0:
    raise FileNotFoundError(
        "No Excel files were found. Check DATA_ROOT and the 2023/2024 folder names."
    )

display(pd.DataFrame({
    "file": [f.name for f in excel_files],
    "folder": [f.parent.name for f in excel_files]
}))

In [ ]:
import re

def read_month_workbook(file_path):
    """Read one monthly S01 workbook and return combined daily data.

    The month and year are taken from the FILE NAME (e.g. S01-03-23.xlsx),
    not from cell A1. In the PEA workbooks A1 is only filled on some sheets
    and sometimes holds a stale date (e.g. 2022 inside a 2023 file), which
    previously caused most days to be skipped or mis-dated.
    """

    match = re.search(r"S01-(\d{2})-(\d{2})", Path(file_path).stem)
    if match is None:
        return pd.DataFrame()

    file_month = int(match.group(1))
    file_year = 2000 + int(match.group(2))

    xls = pd.ExcelFile(file_path, engine="openpyxl")
    daily_frames = []

    for sheet in xls.sheet_names:

        # Sheet names should be day numbers such as 1, 2, ..., 31
        try:
            day = int(str(sheet).strip())
        except ValueError:
            continue

        # Build the real date from the file name + sheet (day) name
        try:
            current_date = pd.Timestamp(year=file_year, month=file_month, day=day)
        except ValueError:
            # Example: sheet 31 in a month with only 30 days
            continue

        raw = xls.parse(sheet, header=None)

        # Expected file structure requires at least 4 rows and 9 columns
        if raw.shape[0] < 4 or raw.shape[1] < 9:
            continue

        # Actual measurements start from row 4 and columns B:I
        temp = raw.iloc[3:, 1:9].copy()

        temp.columns = [
            "time",
            "PVTEMP1",
            "PVTEMP2",
            "AMBTEMP",
            "PYRANO1",
            "PYRANO2",
            "WINDSPEED",
            "P"
        ]

        temp = temp.dropna(subset=["time"])

        # Convert sensor columns to numeric
        numeric_cols = [
            "PVTEMP1", "PVTEMP2", "AMBTEMP",
            "PYRANO1", "PYRANO2", "WINDSPEED", "P"
        ]

        for col in numeric_cols:
            temp[col] = pd.to_numeric(temp[col], errors="coerce")

        # Combine date + time
        def combine_date_time(t):
            try:
                if isinstance(t, pd.Timestamp):
                    t = t.time()
                elif not hasattr(t, "hour"):
                    t = pd.to_datetime(str(t)).time()

                return pd.Timestamp.combine(current_date.date(), t)
            except Exception:
                return pd.NaT

        temp["timestamp"] = temp["time"].apply(combine_date_time)
        temp = temp.dropna(subset=["timestamp"])

        temp = temp[["timestamp"] + numeric_cols]

        daily_frames.append(temp)

    xls.close()

    if not daily_frames:
        return pd.DataFrame()

    return pd.concat(daily_frames, ignore_index=True)


### Load all monthly workbooks

The default rebuilds the raw cache from Excel. Set USE_CACHE to True only to
reuse a cache whose source-file hashes and loader version match the manifest.
The completeness audit separately reopens the source workbooks.


In [ ]:
import hashlib
import json

USE_CACHE = False
manifest_path = CACHE_FILE.with_suffix(".manifest.json")
source_manifest = {
    "loader_version": "filename-sheet-v3",
    "files": [{"name": str(f.relative_to(REPO_ROOT)),
               "sha256": hashlib.sha256(f.read_bytes()).hexdigest()}
              for f in excel_files],
}
cache_matches = (manifest_path.exists() and
                 json.loads(manifest_path.read_text()) == source_manifest)
if USE_CACHE and CACHE_FILE.exists() and cache_matches:
    df_raw = pd.read_csv(CACHE_FILE, parse_dates=["timestamp"])
    print("Loaded validated raw cache:", CACHE_FILE)
else:
    months = [read_month_workbook(file) for file in excel_files]
    months = [frame for frame in months if not frame.empty]
    if not months:
        raise ValueError("No usable measurements found in the source workbooks")
    df_raw = pd.concat(months, ignore_index=True).sort_values("timestamp")
    CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    df_raw.to_csv(CACHE_FILE, index=False)
    manifest_path.write_text(json.dumps(source_manifest, indent=2))
    print("Rebuilt raw cache from Excel:", CACHE_FILE)

df_raw = df_raw.loc[(df_raw.timestamp >= STUDY_START) &
                   (df_raw.timestamp < pd.Timestamp(STUDY_END)+pd.Timedelta(days=1))].copy()
df_raw = df_raw.sort_values("timestamp").reset_index(drop=True)
ts = df_raw.timestamp
if ts.isna().any() or ((ts.dt.minute % 5 != 0) | (ts.dt.second != 0)).any():
    raise ValueError("Invalid/off-grid timestamps require source review")
duplicates = df_raw.duplicated("timestamp", keep=False)
if duplicates.any():
    conflict = df_raw.loc[duplicates].groupby("timestamp").nunique(dropna=False).gt(1).any(axis=1)
    if conflict.any():
        raise ValueError("Conflicting duplicate measurements require source review")
print("Unique days loaded:", ts.dt.normalize().nunique(), "expected 731")
print("Raw rows:", len(df_raw), "duplicate rows:", int(df_raw.duplicated("timestamp").sum()))
print("Start:", ts.min(), "End:", ts.max())
display(df_raw.head())


# 2. Exploratory Data Analysis

We perform EDA to understand the characteristics and quality of the S01 dataset before model development. The analysis covers data coverage, missing values, descriptive statistics, temporal patterns, variable distributions, and correlations. The refreshed outputs support the progress report and the source-quality analysis.

## 2.1 Dataset Structure and Coverage


In [ ]:
print("Dataset information")
print("-" * 50)
print("Rows:", f"{len(df_raw):,}")
print("Columns:", df_raw.shape[1])
print("Start:", df_raw["timestamp"].min())
print("End:", df_raw["timestamp"].max())
print("Duplicate timestamps:", df_raw["timestamp"].duplicated().sum())

expected_5min = pd.Timedelta(minutes=5)
time_diff = df_raw["timestamp"].diff()

print("\nMost common time intervals:")
display(time_diff.value_counts().head(10).to_frame("count"))

print("\nData types:")
display(df_raw.dtypes.to_frame("dtype"))

In [ ]:
from calendar import monthrange

# ── Coverage by year and month (fixed denominator) ──
coverage = (
    df_raw
    .assign(
        year=df_raw["timestamp"].dt.year,
        month=df_raw["timestamp"].dt.month,
        date=df_raw["timestamp"].dt.date
    )
    .groupby(["year", "month"])
    .agg(
        rows=("timestamp", "size"),
        days_present=("date", "nunique"),
        unique_timestamps=("timestamp", "nunique"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max")
    )
    .reset_index()
)

expected_months = pd.period_range(STUDY_START, STUDY_END, freq="M")
month_index = pd.MultiIndex.from_tuples([(m.year,m.month) for m in expected_months], names=["year","month"])
coverage = coverage.set_index(["year","month"]).reindex(month_index).reset_index()
coverage[["rows","days_present","unique_timestamps"]] = coverage[["rows","days_present","unique_timestamps"]].fillna(0).astype(int)
# Fixed: use all study months and calendar days, including entirely absent months
coverage["calendar_days"] = coverage.apply(
    lambda r: monthrange(int(r["year"]), int(r["month"]))[1], axis=1
)
coverage["missing_days"] = coverage["calendar_days"] - coverage["days_present"]
coverage["expected_5min_rows"] = coverage["calendar_days"] * 288
coverage["coverage_%"] = (
    coverage["unique_timestamps"] / coverage["expected_5min_rows"] * 100
).round(2)

print("Monthly Data Coverage (denominator = calendar days × 288)")
print("=" * 60)
display(coverage)

total_calendar = coverage["calendar_days"].sum()
total_present = coverage["days_present"].sum()
total_missing = coverage["missing_days"].sum()
print(f"\nTotal: {total_present} of {total_calendar} calendar days present "
      f"({total_missing} missing)")

coverage.to_csv(
    OUTPUT_DIR / "table_01_monthly_data_coverage.csv",
    index=False
)


## 2.2 Missing Values


In [ ]:
sensor_cols = [
    "PVTEMP1",
    "PVTEMP2",
    "AMBTEMP",
    "PYRANO1",
    "PYRANO2",
    "WINDSPEED",
    "P"
]

missing_table = pd.DataFrame({
    "missing_count": df_raw[sensor_cols].isna().sum(),
    "missing_%": (df_raw[sensor_cols].isna().mean() * 100).round(4)
}).sort_values("missing_%", ascending=False)

display(missing_table)

missing_table.to_csv(
    OUTPUT_DIR / "table_02_missing_values.csv"
)

In [ ]:
# Visualize missing values across all sensor variables
plt.figure(figsize=(10, 5))
plt.bar(missing_table.index, missing_table["missing_%"])
plt.ylabel("Missing values (%)")
plt.xlabel("Variable")
plt.title("Missing Values in the S01 Dataset")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "figure_01_missing_values.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 2.3 Descriptive Statistics


In [ ]:
descriptive_stats = df_raw[sensor_cols].describe().T
descriptive_stats["missing"] = df_raw[sensor_cols].isna().sum()

display(descriptive_stats)

descriptive_stats.to_csv(
    OUTPUT_DIR / "table_03_descriptive_statistics.csv"
)

## 2.4 Time-Series Visualization

A sample week of 5-minute power generation data is shown to illustrate the daily solar cycle present in the dataset.

In [ ]:
# Automatically select the first 7 complete calendar days available
plot_start = df_raw["timestamp"].min().normalize()
plot_end = plot_start + pd.Timedelta(days=7)

sample_week = df_raw[
    (df_raw["timestamp"] >= plot_start) &
    (df_raw["timestamp"] < plot_end)
]

plt.figure(figsize=(14, 5))
plt.plot(sample_week["timestamp"], sample_week["P"])
plt.xlabel("Time")
plt.ylabel("Power (kW)")
plt.title(f"S01 Solar Power Generation: {plot_start.date()} to {(plot_end - pd.Timedelta(days=1)).date()}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "figure_02_power_timeseries_sample_week.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 2.5 Mean Daily Solar Power Profile

The mean power at each time of day is computed separately for 2023 and 2024 to observe the typical daily generation pattern and any year-to-year differences.

In [ ]:
eda_profile = df_raw.copy()

eda_profile["year"] = eda_profile["timestamp"].dt.year
eda_profile["time_of_day"] = eda_profile["timestamp"].dt.strftime("%H:%M")

daily_profile = (
    eda_profile
    .groupby(["year", "time_of_day"], as_index=False)["P"]
    .mean()
)

plt.figure(figsize=(12, 5))

for year in sorted(daily_profile["year"].unique()):
    temp = daily_profile[daily_profile["year"] == year]
    plt.plot(
        np.arange(len(temp)),
        temp["P"],
        label=str(year)
    )

# Tick every 3 hours = 36 samples at 5-min resolution
tick_positions = np.arange(0, 288, 36)
tick_labels = [f"{h:02d}:00" for h in range(0, 24, 3)]

plt.xticks(tick_positions, tick_labels)
plt.xlabel("Time of Day")
plt.ylabel("Mean Power (kW)")
plt.title("Mean Daily Solar Power Profile of S01")
plt.legend()
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "figure_03_mean_daily_power_profile.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 2.6 Monthly Energy Pattern

Since power (P) is recorded in kW at 5-minute intervals, the approximate energy for each record is P × (5/60) kWh. Monthly totals show the seasonal variation across the two study years.

In [ ]:
energy_df = df_raw[["timestamp", "P"]].copy()

# Do not fill missing target values; only valid measurements contribute
energy_df["energy_kWh"] = energy_df["P"] * (5 / 60)
energy_df["month"] = energy_df["timestamp"].dt.to_period("M").astype(str)

monthly_energy = (
    energy_df
    .groupby("month", as_index=False)["energy_kWh"]
    .sum()
)

monthly_energy["energy_MWh"] = monthly_energy["energy_kWh"] / 1000

display(monthly_energy[["month", "energy_MWh"]])

monthly_energy.to_csv(
    OUTPUT_DIR / "table_04_monthly_energy.csv",
    index=False
)

plt.figure(figsize=(13, 5))
plt.bar(monthly_energy["month"], monthly_energy["energy_MWh"])
plt.xlabel("Month")
plt.ylabel("Approximate Energy (MWh)")
plt.title("Monthly Solar Energy of S01")
plt.xticks(rotation=60)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "figure_04_monthly_energy.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 2.7 Distribution of Power Generation

Solar power data contain many zero values at night. We show two distributions: one for all 24-hour observations and one limited to daytime hours (06:00–18:00). This also illustrates why percentage-based metrics such as MAPE become unstable near zero generation.

In [ ]:
hour_decimal = (
    df_raw["timestamp"].dt.hour +
    df_raw["timestamp"].dt.minute / 60
)

daylight_mask = (hour_decimal >= 6) & (hour_decimal < 18)

plt.figure(figsize=(10, 5))
plt.hist(df_raw["P"].dropna(), bins=60)
plt.xlabel("Power (kW)")
plt.ylabel("Frequency")
plt.title("Distribution of S01 Power Generation: Full Day")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "figure_05_power_distribution_full_day.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(df_raw.loc[daylight_mask, "P"].dropna(), bins=60)
plt.xlabel("Power (kW)")
plt.ylabel("Frequency")
plt.title("Distribution of S01 Power Generation: 06:00–18:00")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "figure_06_power_distribution_daylight.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Full-day skewness:", round(df_raw["P"].skew(), 4))
print(
    "Daylight skewness:",
    round(df_raw.loc[daylight_mask, "P"].skew(), 4)
)

## 2.8 Outlier Examination

Boxplots are used to visually inspect each key variable for unusual observations. Points beyond the whiskers indicate values that fall outside the typical interquartile range.

In [ ]:
key_vars = ["P", "PYRANO1", "AMBTEMP", "WINDSPEED"]

fig, axes = plt.subplots(1, len(key_vars), figsize=(16, 5))

for ax, col in zip(axes, key_vars):
    ax.boxplot(df_raw[col].dropna(), vert=True)
    ax.set_title(col)
    ax.set_ylabel(col)
    ax.set_xticklabels([""])

plt.suptitle("Boxplot of Key Variables in S01 Dataset", fontsize=14)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "figure_06_boxplots.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Count observations beyond the IQR fences for each variable
print("Observations beyond IQR fences (potential outliers):")
print("-" * 50)
for col in key_vars:
    vals = df_raw[col].dropna()
    q1 = vals.quantile(0.25)
    q3 = vals.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((vals < lower) | (vals > upper)).sum()
    print(f"  {col:12s}: {n_outliers:,} observations ({n_outliers / len(vals) * 100:.2f}%)")

## 2.9 Data Distribution and Skewness

We compute the skewness of each variable to understand the shape of its distribution. Positive skewness indicates a right-tailed distribution, while values near zero suggest approximate symmetry.

In [ ]:
skewness = df_raw[sensor_cols].skew().round(3)
kurtosis = df_raw[sensor_cols].kurtosis().round(3)

dist_summary = pd.DataFrame({
    "skewness": skewness,
    "kurtosis": kurtosis
})

display(dist_summary)

dist_summary.to_csv(
    OUTPUT_DIR / "table_04_skewness_kurtosis.csv"
)

print("\nInterpretation:")
for col in sensor_cols:
    sk = skewness[col]
    if abs(sk) < 0.5:
        shape = "approximately symmetric"
    elif sk > 0:
        shape = "right-skewed (positively skewed)"
    else:
        shape = "left-skewed (negatively skewed)"
    print(f"  {col:12s}: {sk:+.3f} — {shape}")

## 2.10 Solar Irradiance vs. Power Generation

We investigate the relationship between measured solar irradiance (PYRANO1) and PV power output. A reproducible sample is used for plotting when the dataset is large.

In [ ]:
scatter_df = df_raw[["PYRANO1", "P"]].dropna()

if len(scatter_df) > 20000:
    scatter_df = scatter_df.sample(
        20000,
        random_state=RANDOM_STATE
    )

plt.figure(figsize=(8, 6))
plt.scatter(
    scatter_df["PYRANO1"],
    scatter_df["P"],
    alpha=0.2,
    s=10
)
plt.xlabel("Solar Irradiance - PYRANO1 (W/m²)")
plt.ylabel("Power (kW)")
plt.title("Solar Irradiance vs Power Generation")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "figure_07_irradiance_vs_power.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

irradiance_corr = df_raw[["PYRANO1", "P"]].corr().iloc[0, 1]
print("Pearson correlation between PYRANO1 and P:", round(irradiance_corr, 4))

## 2.11 Correlation Matrix


In [ ]:
corr_matrix = df_raw[sensor_cols].corr()

display(corr_matrix.round(3))

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix.values, aspect="auto")

ax.set_xticks(np.arange(len(corr_matrix.columns)))
ax.set_yticks(np.arange(len(corr_matrix.index)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right")
ax.set_yticklabels(corr_matrix.index)

for i in range(len(corr_matrix.index)):
    for j in range(len(corr_matrix.columns)):
        ax.text(
            j, i,
            f"{corr_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

fig.colorbar(im, ax=ax, label="Correlation")
ax.set_title("Correlation Matrix of S01 Variables")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "figure_08_correlation_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 2.12 Preliminary Findings Summary

A concise numerical summary of the key findings from the EDA, used as reference when writing the Preliminary Results section of the proposal.

In [ ]:
valid_p = df_raw["P"].dropna()

summary = {
    "Study period": (
        f"{df_raw['timestamp'].min().date()} to "
        f"{df_raw['timestamp'].max().date()}"
    ),
    "5-min records": len(df_raw),
    "Unique calendar days": df_raw["timestamp"].dt.date.nunique(),
    "Duplicate timestamps": int(df_raw["timestamp"].duplicated().sum()),
    "Missing P values": int(df_raw["P"].isna().sum()),
    "Mean power (kW)": valid_p.mean(),
    "Maximum observed power (kW)": valid_p.max(),
    "Power skewness": valid_p.skew(),
    "PYRANO1-P correlation": irradiance_corr
}

summary_df = pd.DataFrame(
    summary.items(),
    columns=["Item", "Value"]
)

display(summary_df)

summary_df.to_csv(
    OUTPUT_DIR / "table_05_preliminary_summary.csv",
    index=False
)

print("\n--- Proposal writing helper ---")
print(
    f"The dataset covers {summary['Study period']} and contains "
    f"{summary['5-min records']:,} five-minute records."
)
print(
    f"The observed correlation between PYRANO1 and power generation is "
    f"{summary['PYRANO1-P correlation']:.4f}."
)
print(
    f"The maximum observed power is "
    f"{summary['Maximum observed power (kW)']:.2f} kW."
)
print("\nUse these statements only after checking the EDA figures above.")

# 3. Data Preprocessing

The original 5-minute measurements are aggregated into 15-minute intervals to align with PEA's standard metering resolution. For each 15-minute window, the mean of the available 5-minute readings is computed.

For the target variable (power generation), an interval is set to missing if fewer than three valid 5-minute measurements are present, since a reliable actual value is required for model evaluation.

In [ ]:
df = df_raw.copy()
df = df.drop_duplicates(subset="timestamp", keep="first")
df = df.sort_values("timestamp")
df = df.set_index("timestamp")

numeric_cols = [
    "PVTEMP1",
    "PVTEMP2",
    "AMBTEMP",
    "PYRANO1",
    "PYRANO2",
    "WINDSPEED",
    "P"
]

# Note: raw P is kept as is (small negative values near sunrise/sunset are preserved).
# Clipping to the 0-6,000 kW operating range is decided at the modelling stage.

# Mean values in each 15-min window
target_index = pd.date_range(STUDY_START, pd.Timestamp(STUDY_END)+pd.Timedelta(days=1), freq="15min", inclusive="left")
df_15 = df[numeric_cols].resample("15min").mean().reindex(target_index)

# Count how many valid 5-min observations contributed to each 15-min P value
p_count = df["P"].resample("15min").count().reindex(target_index, fill_value=0)
df_15["P_count_5min"] = p_count

# Strict target: need all 3 expected measurements
df_15.loc[df_15["P_count_5min"] < 3, "P"] = np.nan

print("15-min rows:", f"{len(df_15):,}")
print("Incomplete target intervals:", int((df_15["P_count_5min"] < 3).sum()))
print("Missing 15-min target P:", int(df_15["P"].isna().sum()))

display(df_15.head())

# Round to avoid floating-point artifacts such as 15.316666666666668
# (Excel can flag these long values as text)
df_15 = df_15.round(4)

df_15.to_csv(
    OUTPUT_DIR / "S01_2023_2024_15min_processed.csv"
)


### 15-Minute Completeness by Day

A complete day should contain 96 valid 15-minute target intervals. We check how many days meet this criterion.

In [ ]:
daily_completeness = (
    df_15["P"]
    .notna()
    .groupby(df_15.index.date)
    .sum()
    .rename("valid_intervals")
    .to_frame()
)

daily_completeness["complete_96"] = (
    daily_completeness["valid_intervals"] == 96
)

print(
    "Complete days:",
    int(daily_completeness["complete_96"].sum()),
    "/",
    len(daily_completeness)
)

display(daily_completeness["valid_intervals"].value_counts().sort_index().tail(15))

In [ ]:
#  Day completeness with flags
daily_completeness["low_completeness"] = daily_completeness["valid_intervals"] < 80

complete_days = int(daily_completeness["complete_96"].sum())
low_days = int(daily_completeness["low_completeness"].sum())
partial_days = int(
    (~daily_completeness["complete_96"] & ~daily_completeness["low_completeness"]).sum()
)
total_days = len(daily_completeness)

print("Day-level completeness summary")
print("=" * 50)
print(f"  Complete (96/96):     {complete_days}")
print(f"  Partial (80–95):      {partial_days}")
print(f"  Low (<80 intervals):  {low_days}")
print(f"  Total days:           {total_days}")

# Show all non-complete days
incomplete = daily_completeness[~daily_completeness["complete_96"]].copy()
incomplete.index = pd.to_datetime(incomplete.index)
print(f"\n{len(incomplete)} days with fewer than 96 valid intervals:")
display(incomplete.sort_values("valid_intervals"))

# ── Calendar heatmap: valid P intervals per day ──
import matplotlib.dates as mdates

heatmap_df = daily_completeness.copy()
heatmap_df.index = pd.to_datetime(heatmap_df.index)

fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(
    heatmap_df.index,
    heatmap_df["valid_intervals"],
    c=heatmap_df["valid_intervals"],
    cmap="RdYlGn",
    vmin=0, vmax=96,
    s=8,
    edgecolors="none"
)
ax.axhline(96, color="green", linewidth=0.5, linestyle="--", alpha=0.5)
ax.axhline(80, color="red", linewidth=0.5, linestyle="--", alpha=0.5)
ax.set_xlabel("Date")
ax.set_ylabel("Valid 15-min P intervals")
ax.set_title("Daily Completeness of Target Variable (P)")
ax.set_ylim(-2, 100)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure_09_daily_completeness.png", dpi=150)
plt.show()

## 3 Sensor quality rules

Weather flags use the existing study heuristics. Frozen-sensor detection uses
trailing run length and masks only the 12th consecutive reading onward, without
changing earlier observations. P is not imputed or masked by weather rules.
Threshold definitions and affected-row counts are exported for review.


In [ ]:
from src.data_quality import clean_sensor_observations
df = df_raw.drop_duplicates("timestamp").sort_values("timestamp").set_index("timestamp")
df, summary_df = clean_sensor_observations(df, output_dir=OUTPUT_DIR)
display(summary_df)


### Aggregate cleaned observations

Require three valid 5-minute P readings for each 15-minute target. Reindex to the fixed study calendar.


In [ ]:
#  Resample to 15-minute intervals (on masked data)
numeric_cols = ["PVTEMP1", "PVTEMP2", "AMBTEMP", "PYRANO1", "PYRANO2", "WINDSPEED", "P"]

target_index = pd.date_range(STUDY_START, pd.Timestamp(STUDY_END)+pd.Timedelta(days=1), freq="15min", inclusive="left")
df_15 = df[numeric_cols].resample("15min").mean().reindex(target_index)

# Count how many valid 5-min P observations contributed to each 15-min interval
p_count = df["P"].resample("15min").count().reindex(target_index, fill_value=0)
df_15["P_count_5min"] = p_count

# Strict target: need all 3 expected measurements
df_15.loc[df_15["P_count_5min"] < 3, "P"] = np.nan

# Carry flag columns forward 
flag_cols = [c for c in df.columns if c.startswith("flag_")]
for fc in flag_cols:
    df_15[fc] = df[fc].resample("15min").max()

print("15-min rows:", f"{len(df_15):,}")
print("Missing 15-min target P:", int(df_15["P"].isna().sum()))
print(f"Flag columns carried: {flag_cols}")

df_15 = df_15.round(4)
df_15.to_csv(OUTPUT_DIR / "S01_2023_2024_15min_processed.csv")

display(df_15.head())


### 15-Minute Completeness by Day

A complete day should contain 96 valid 15-minute target intervals. We will check how many days meet this after sensor masking processes.

In [ ]:
# Day completeness (re-run on cleaned data)
daily_completeness = (
    df_15["P"]
    .notna()
    .groupby(df_15.index.date)
    .sum()
    .rename("valid_intervals")
    .to_frame()
)

daily_completeness["complete_96"] = daily_completeness["valid_intervals"] == 96
daily_completeness["low_completeness"] = daily_completeness["valid_intervals"] < 80

complete_days = int(daily_completeness["complete_96"].sum())
low_days = int(daily_completeness["low_completeness"].sum())
partial_days = int(
    (~daily_completeness["complete_96"] & ~daily_completeness["low_completeness"]).sum()
)
total_days = len(daily_completeness)

print("Day-level completeness summary (after sensor masking)")
print("=" * 50)
print(f"  Complete (96/96):     {complete_days}")
print(f"  Partial (80–95):      {partial_days}")
print(f"  Low (<80 intervals):  {low_days}")
print(f"  Total days:           {total_days}")


incomplete = daily_completeness[~daily_completeness["complete_96"]].copy()
incomplete.index = pd.to_datetime(incomplete.index)
print(f"\n{len(incomplete)} days with fewer than 96 valid intervals:")
display(incomplete.sort_values("valid_intervals"))

# ── Calendar heatmap ──
import matplotlib.dates as mdates

heatmap_df = daily_completeness.copy()
heatmap_df.index = pd.to_datetime(heatmap_df.index)

fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(
    heatmap_df.index,
    heatmap_df["valid_intervals"],
    c=heatmap_df["valid_intervals"],
    cmap="RdYlGn",
    vmin=0, vmax=96,
    s=8,
    edgecolors="none"
)
ax.axhline(96, color="green", linewidth=0.5, linestyle="--", alpha=0.5)
ax.axhline(80, color="red", linewidth=0.5, linestyle="--", alpha=0.5)
ax.set_xlabel("Date")
ax.set_ylabel("Valid 15-min P intervals")
ax.set_title("Daily Completeness of Target Variable (P) — After Sensor Masking")
ax.set_ylim(-2, 100)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure_09_daily_completeness.png", dpi=150)
plt.show()

In [ ]:
from src.data_quality import run_completeness_audit
completeness_audit, monthly_target, coverage_report = run_completeness_audit(
    df_raw,daily_completeness,coverage,excel_files,OUTPUT_DIR,
    study_start=STUDY_START,study_end=STUDY_END,
)
display(completeness_audit.loc[~completeness_audit.complete_96])


### Day eligibility and paired sensors

Days with fewer than 80/96 valid P targets are excluded under the study policy.
For retained partial days, only valid targets are fitted/scored. Dates previously
listed as possible outages are retained for review unless they fail this rule or
have a documented manual exclusion. Weather flags alone do not prove an outage.
The exported daily log records observed counts, flags, eligibility, and reasons.


In [ ]:
from src.data_quality import apply_day_quality_policy
df_15["irradiance"] = df_15[["PYRANO1","PYRANO2"]].mean(axis=1)
df_15["pv_temp"] = df_15[["PVTEMP1","PVTEMP2"]].mean(axis=1)
df_15 = df_15.drop(columns=["PYRANO1","PYRANO2","PVTEMP1","PVTEMP2"])
df_15, day_quality_log = apply_day_quality_policy(
    df_15, output_dir=OUTPUT_DIR, min_valid_targets=80,
    manual_exclusions=MANUAL_EXCLUSIONS,
)
EXCLUDE_DATES = day_quality_log.index[day_quality_log.excluded]
feature_cols = ["AMBTEMP","WINDSPEED","irradiance","pv_temp"]
print("Excluded days:", len(EXCLUDE_DATES))
print("Retained dates:", int((~day_quality_log.excluded).sum()))
display(day_quality_log.loc[day_quality_log.excluded | day_quality_log.legacy_review_date])


## 4 Chronological train and test split

The hold-out contains the last 20% of dates retained under the documented quality policy. Date ranges are recalculated on each run; old results must not be reused.


In [ ]:
# Chronological train/test split on non-excluded days

# เก็บ raw clean data แยกไว้ เพื่อให้ walk-forward impute ใหม่ในแต่ละ fold
df_clean_raw = df_15[~df_15["excluded"]].copy()
df_clean = df_clean_raw.copy()

all_dates = pd.Series(df_clean.index.normalize().unique()).sort_values().reset_index(drop=True)
n_days = len(all_dates)

split_idx = int(n_days * 0.8)
train_end_date = all_dates.iloc[split_idx - 1]
test_start_date = all_dates.iloc[split_idx]

df_train_raw = df_clean_raw[df_clean_raw.index.normalize() <= train_end_date].copy()
df_test_raw = df_clean_raw[df_clean_raw.index.normalize() >= test_start_date].copy()
df_train = df_train_raw.copy()
df_test = df_test_raw.copy()

print("Chronological Train/Test Split")
print("=" * 50)
print(f"  Total usable days:  {n_days}")
print(f"  Train days:         {df_train.index.normalize().nunique()} ({df_train.index.normalize().nunique()/n_days:.1%})")
print(f"  Test days:          {df_test.index.normalize().nunique()} ({df_test.index.normalize().nunique()/n_days:.1%})")
print(f"  Train period:       {df_train.index.min().date()} to {df_train.index.max().date()}")
print(f"  Test period:        {df_test.index.min().date()} to {df_test.index.max().date()}")
print(f"  Train rows:         {len(df_train):,}")
print(f"  Test rows:          {len(df_test):,}")

train_dates = set(df_train.index.normalize())
test_dates = set(df_test.index.normalize())
assert train_dates.isdisjoint(test_dates), "LEAKAGE: train and test dates overlap!"
print("\nNo date overlap between train and test sets")

### Training-only feature imputation

Build historical features from unfilled observations. Fit slot medians inside each training fold, with a training-median fallback and a fixed zero fallback only for an entirely absent feature. Targets P are never imputed. Ridge also fits its scaler on training data only.


In [ ]:
# Keep raw train/test frames until feature construction is complete

# สำคัญ: ยังไม่ impute ที่จุดนี้ เพราะการ impute ก่อน CV จะทำให้สถิติจากอนาคต
# รั่วเข้า validation fold ได้ โดย imputer จะถูก fit ใหม่ในแต่ละ CV fold ภายหลัง
feature_cols = ["AMBTEMP", "WINDSPEED", "irradiance", "pv_temp"]
assert "P" not in feature_cols, "P must not be in the imputation list"
df_train = df_train_raw.copy()
df_test = df_test_raw.copy()
print("Raw train/test retained; fold-safe imputation is deferred to model pipelines.")
print(f"Remaining NaN in target P: train={df_train['P'].isna().sum()}, test={df_test['P'].isna().sum()}")

In [ ]:
from src.modeling import CalendarDayTimeSeriesSplit
tscv = CalendarDayTimeSeriesSplit(n_splits=5, gap_days=1)
print("CV: five expanding folds; whole calendar dates; one calendar-day embargo")


## 5 Feature engineering

Use only observations from D-1 or earlier and known calendar features. Daily energy is in kWh (sum of 96 quarter-hour power values times 0.25 h); incomplete daily energy remains missing. Seven-day aggregates require at least three valid historical values.


In [ ]:
from src.data_quality import build_forecast_features
build_features = build_forecast_features
df_full_imputed = pd.concat([df_train_raw,df_test_raw]).sort_index()  # raw, despite legacy name
df_features = build_features(df_full_imputed)
df_train_feat = df_features.loc[df_features.index.normalize() <= train_end_date].copy()
df_test_feat = df_features.loc[df_features.index.normalize() >= test_start_date].copy()
target_col = "P"
feature_names = [c for c in df_features if c != target_col]
print("Features:",feature_names)


In [ ]:
# Drop rows where historical power features are unavailable (first 7 days/gaps)

train_before = len(df_train_feat)
test_before = len(df_test_feat)

power_history_features = [
    "P_lag1", "P_lag2", "P_lag7", "P_roll7",
    "energy_roll7_mean", "energy_roll7_std",
]
df_train_feat = df_train_feat.dropna(subset=power_history_features)
df_test_feat = df_test_feat.dropna(subset=power_history_features)
# weather-lag NaN จะถูกจัดการโดย fold-safe imputer ใน modeling pipeline

# Also drop rows where target P is NaN (excluded from loss)
df_train_feat = df_train_feat.dropna(subset=[target_col])
df_test_feat = df_test_feat.dropna(subset=[target_col])

print(f"After dropping NaN features and missing P:")
print(f"  Train: {train_before} -> {len(df_train_feat)} ({train_before - len(df_train_feat)} dropped)")
print(f"  Test:  {test_before} -> {len(df_test_feat)} ({test_before - len(df_test_feat)} dropped)")
print(f"\n  Train days: {df_train_feat.index.normalize().nunique()}")
print(f"  Test days:  {df_test_feat.index.normalize().nunique()}")
print(f"\n  Train period: {df_train_feat.index.min().date()} to {df_train_feat.index.max().date()}")
print(f"  Test period:  {df_test_feat.index.min().date()} to {df_test_feat.index.max().date()}")

# Split into X and y
X_train = df_train_feat[feature_names]
y_train = df_train_feat[target_col]
X_test = df_test_feat[feature_names]
y_test = df_test_feat[target_col]

# ตรวจจำนวน interval ต่อวันหลังกรอง feature/target เพื่อแยก complete กับ incomplete forecast days
from src.data_quality import summarize_forecast_day_completeness
test_forecast_day_summary = summarize_forecast_day_completeness(
    df_test_feat.index,
    target_values=df_test_feat[target_col],
    output_dir=OUTPUT_DIR,
    file_name="table_14_test_forecast_day_completeness.csv",
)
print("Test forecast-day completeness")
display(test_forecast_day_summary)
print("Complete days:", int(test_forecast_day_summary["target_complete"].sum()))
print("Incomplete days:", int((~test_forecast_day_summary["target_complete"]).sum()))

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

In [ ]:
from src.data_quality import run_feature_availability_audit
feature_availability_audit = run_feature_availability_audit(
    df_full_imputed, df_features, feature_names, OUTPUT_DIR,
)


## 6 Simple forecasting baselines

Compare previous-day persistence, previous-week persistence, and a historical monthly time-slot mean. Historical profiles use training observations only, with a training time-slot fallback for unseen months. Every method is scored on the same timestamps and uses prediction clipping to 0-6,000 kW. Observed zero targets are retained.


In [ ]:
from src.modeling import evaluate_baselines
from src.data_quality import summarize_forecast_day_completeness


In [ ]:
baseline_results, baseline_predictions = evaluate_baselines(
    df_train_raw, df_test_feat, PLANT_CAPACITY_KW,
)
results_df = baseline_results.copy()
persist1_pred = baseline_predictions["Persistence (t-1)"]
persist7_pred = baseline_predictions["Persistence (t-7)"]
hist_avg_pred = baseline_predictions["Historical Average"]
y_actual = y_test.to_numpy()
test_index = df_test_feat.index
evaluation_day_summary = summarize_forecast_day_completeness(
    test_index,y_test,OUTPUT_DIR,"table_15_evaluation_day_completeness.csv",
)
display(baseline_results)
baseline_results.to_csv(OUTPUT_DIR/"table_08_baseline_results.csv",index=False)


## 7 Monthly baseline evaluation

Use earlier data for training and evaluate each subsequent calendar month after six initial months. The same feature-availability mask is used for all methods. Record complete, partial, and unscored calendar days. Seasonal summaries are unweighted means of monthly metrics, not pooled interval-level errors.


In [ ]:
from src.modeling import walk_forward_baselines
wf_results = walk_forward_baselines(
    df_clean_raw, feature_names, build_features, OUTPUT_DIR,
    min_train_months=6, capacity_kw=PLANT_CAPACITY_KW,
)
display(wf_results)


In [ ]:
# Seasonal summary

season_summary = wf_results.groupby("Season")[
    ["Persist(t-1) MAE", "Persist(t-7) MAE", "HistAvg MAE"]
].mean().round(2)

season_summary = season_summary.reindex(["Hot", "Rainy", "Cool"])
print("Average MAE by Thai Season")
print("=" * 50)
display(season_summary)

# Monthly MAE chart with season coloring
season_colors = {"Hot": "#e74c3c", "Rainy": "#3498db", "Cool": "#2ecc71"}

fig, ax = plt.subplots(figsize=(14, 5))

x = range(len(wf_results))
bar_width = 0.25

for idx, (col, label) in enumerate([
    ("Persist(t-1) MAE", "Persistence (t-1)"),
    ("Persist(t-7) MAE", "Persistence (t-7)"),
    ("HistAvg MAE", "Historical Average")
]):
    offsets = [i + idx * bar_width for i in x]
    colors = [season_colors[s] for s in wf_results["Season"]]
    ax.bar(offsets, wf_results[col], bar_width, label=label, alpha=0.4 + idx * 0.25,
           color=colors, edgecolor="white", linewidth=0.5)

ax.set_xticks([i + bar_width for i in x])
ax.set_xticklabels(wf_results["Month"], rotation=45, ha="right")
ax.set_ylabel("MAE (kW)")
ax.set_title("Walk-Forward Monthly MAE by Baseline (colored by Thai season)")

from matplotlib.patches import Patch
season_patches = [Patch(color=c, label=s) for s, c in season_colors.items()]
ax.legend(handles=season_patches, loc="upper right", title="Season")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "figure_10_walkforward_monthly_mae.png", dpi=150)
plt.show()

## 8 Ridge Regression and XGBoost

The linear baseline is Ridge Regression (L2-regularized Linear Regression), alpha=1.0. L2 controls coefficient sizes for overlapping lag/rolling features. The regularization is part of the linear baseline training objective. XGBoost is evaluated with feature ablation and 10th-to-90th percentile prediction intervals. Monthly parameters are tuned independently on the initial six months only.


In [ ]:
from src.modeling import fit_advanced_models, plot_selected_forecast_days, walk_forward_with_models, plot_walk_forward_model_mae

advanced = fit_advanced_models(
    X_train, y_train, X_test, y_test, test_index, feature_names,
    baseline_results=baseline_results.copy(), output_dir=OUTPUT_DIR,
    tscv=tscv, impute_feature_columns=feature_names,
    plant_capacity_kw=PLANT_CAPACITY_KW, random_state=RANDOM_STATE,
    search_n_iter=XGB_SEARCH_ITER,
)
display(advanced["ridge_top_coefficients"][["Feature","Coefficient"]])
display(pd.DataFrame([advanced["best_params"]]))
results_df = advanced["results_df"]
display(results_df)
display(advanced["interval_metrics"])
selected_days = plot_selected_forecast_days(
    test_index, y_test.to_numpy(), advanced["y_pred_xgb"],
    advanced["y_low"],advanced["y_high"],persist1_pred,
    df_test_feat["irr_lag1"].to_numpy(),OUTPUT_DIR,
)

# No hold-out best_params are passed to the earlier monthly windows.
wf_models = walk_forward_with_models(
    df_clean_raw, feature_names, build_features, output_dir=OUTPUT_DIR,
    min_train_months=6, random_state=RANDOM_STATE,
    search_n_iter=XGB_SEARCH_ITER, capacity_kw=PLANT_CAPACITY_KW,
)
assert (pd.to_datetime(wf_models["Parameter selection end"]) < pd.to_datetime(wf_models["Test start"])).all()
assert wf_models["Test intervals"].to_list() == wf_results["Test intervals"].to_list()
display(wf_models)
plot_walk_forward_model_mae(wf_models,OUTPUT_DIR)
seasonal_models = wf_models.groupby("Season")[[
    "Persistence (t-1) MAE","Historical Average MAE","Ridge MAE","XGBoost MAE"
]].mean().reindex(["Hot","Rainy","Cool"])
print("Seasonal MAE: unweighted mean of monthly MAE (kW)")
display(seasonal_models)
seasonal_models.to_csv(OUTPUT_DIR/"table_13_seasonal_models.csv")


## 9 Audit scope and result interpretation

The source audit compares daily source rows and missing-P counts with loaded data; it is not an exact value-by-value validation. Timestamp coverage, complete-target days, retained policy days, and scored forecast days are separate counts. The earlier 548/731 result comprises 548 complete and 5 partial loaded days, plus 178 dates absent from the earlier loaded dataset. Do not interpret the change as newly collected measurements.

Feature audit PASS checks lag/rolling/calendar definitions. Causal cleaning and chronological parameter selection are separate safeguards. Recompute all reported metrics after running this revision.


In [ ]:
from src.data_quality import run_feature_availability_audit
feature_availability_audit = run_feature_availability_audit(
    df_full_imputed, df_features, feature_names, OUTPUT_DIR,
)
print("Report counts from this run:")
print("Calendar dates:",len(completeness_audit))
print("Complete target dates:",int(completeness_audit.complete_96.sum()))
print("Retained policy dates:",int((~day_quality_log.excluded).sum()))
print("Scored hold-out intervals:",len(y_test))
print("Full-day and daytime scores must be interpreted separately.")
